# MIRO AI -- Benchmark de precision

objectif : heberger un LLM local sur le GPU gratuit de Kaggle, l'exposer via un tunnel public, puis pointer le pipeline MIRO AI

## 1. Verifier le GPU

In [25]:
!nvidia-smi

Tue Sep 29 06:19:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Installer Ollama

In [26]:
# zstd requis par l'installeur pour l'extraction ; pciutils (lspci) requis pour la détection GPU 
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                       0.0%                                                   27.7%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [27]:
# 2bis. configuration OLLAMA_MODELS
import os, glob

OLLAMA_MODELS_DIR = "/kaggle/working/ollama_models"
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
READONLY_MODELS = False
print(f"{OLLAMA_MODELS_DIR} (téléchargement nécessaire ci-dessous).")

os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR

/kaggle/working/ollama_models (téléchargement nécessaire ci-dessous).


## 3. Demarrer le serveur Ollama en arrière-plan

In [28]:
  # 2ter. Configuration Ollama pour le benchmark Qwen3

  import os

  os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
  os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
  os.environ["OLLAMA_LOAD_TIMEOUT"] = "30m"

  print("Configuration Ollama active avant le lancement du serveur:")
  print("OLLAMA_KEEP_ALIVE=-1")
  print("OLLAMA_MAX_LOADED_MODELS=1")
  print("OLLAMA_LOAD_TIMEOUT=30m")
  print("Le modele utilisé est defini dans la cellule de téléchargement.")

Configuration Ollama active avant le lancement du serveur:
OLLAMA_KEEP_ALIVE=-1
OLLAMA_MAX_LOADED_MODELS=1
OLLAMA_LOAD_TIMEOUT=30m
Le modele utilisé est defini dans la cellule de téléchargement.


In [29]:
import subprocess, time, requests

# Tuer tout ollama serve deja lancé sinon le nouveau process echoue au bind du port 11434 silencieusement.
# Rend la cellule rejouable telle quelle
!pkill -f "ollama serve" || true
time.sleep(2)  # laisser le temps au port de se liberer avant le nouveau bind

# OLLAMA_ORIGINS=* : desactive le controle CORS sur le header Origin. Ne couvre PAS le rejet
# 403 du header Host (protection anti DNS-rebinding distincte, whitelist localhost/127.0.0.1
# uniquement) 
os.environ["OLLAMA_ORIGINS"] = "*"

# OLLAMA_CONTEXT_LENGTH=16384 : la fenetre de contexte par defaut d'Ollama (4096 tokens,
# prompt+completion CUMULES) coupe la generation en cours de synthese sur les questions
# multi-hop (observations de plusieurs outils CI concatenes dans le prompt) 
os.environ["OLLAMA_CONTEXT_LENGTH"] = "16384"

os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env=os.environ,  # necessaire pour que OLLAMA_MODELS et OLLAMA_KEEP_ALIVE soient pris en compte
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print("Ollama prêt.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama n'a pas demarré à temps -- verifier les logs du processus (ollama_proc.stderr.read()).")

Ollama prêt.


In [30]:
# 3bis. Verifier qu'Ollama a bien detecté le GPU
!ollama --version
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

]11;?\ollama version is 0.34.4
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB
Tesla T4, 0 MiB, 15360 MiB


## 4. Télécharger les modèles

In [31]:
#MODEL = "gpt-oss:20b"

# Alternative de comparaison :
MODEL = "qwen3:14b"

if not READONLY_MODELS:
    !ollama pull {MODEL}
else:
    print(f"Mode B : {MODEL} supposé déja présent dans {OLLAMA_MODELS_DIR} -- verification via 'ollama list' :")
    !ollama list

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling a8cc1361f314: 100% ▕██████████████████▏ 9.3 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 78b3b822087d: 100% ▕██████████████████▏  488 B                         
verifying sha256 digest 
writing manifest 
success 


In [32]:
# 4ter. Télécharger le modele d'embedding
# nomic-embed-text (~274 Mo) -- meme serveur Ollama que le LLM de chat, aucun conflit VRAM
# reel avec gpt-oss:20b (~13 Go) sur les 15-16 Go d'un T4.
EMBED_MODEL = "nomic-embed-text"

if not READONLY_MODELS:
    !ollama pull {EMBED_MODEL}
else:
    print(f"{EMBED_MODEL} supposé deja présent dans {OLLAMA_MODELS_DIR}.")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


## 5. Préchauffage + sanity check tool-calling

In [33]:
# 5. Prechauffer le modele en VRAM avant le sanity check
# Ollama instancie un runner different selon la presence de 'tools'
import requests, time

_t0 = time.time()
requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "hi"}],
        "tools": [{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Renvoie la météo pour une ville donnée.",
                "parameters": {
                    "type": "object",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"],
                },
            },
        }],
        "stream": False,
    },
    timeout=600,
)
print(f"Modèle chargé en VRAM ({time.time() - _t0:.1f}s).")

Modèle chargé en VRAM (110.1s).


In [34]:
# 5bis. Sanity check local -- verifier que le tool-calling natif fonctionne

import requests, json as _json

test_payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Quel temps fait-il a Paris ?"}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Renvoie la meteo pour une ville donnée.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }],
    "stream": False,
}

resp = requests.post("http://localhost:11434/v1/chat/completions", json=test_payload, timeout=300)  # large par securité, le modèle est déja préchauffé
resp.raise_for_status()
data = resp.json()
tool_calls = data["choices"][0]["message"].get("tool_calls")
print("Tool calls recus :", _json.dumps(tool_calls, indent=2, ensure_ascii=False))
assert tool_calls, "tool-calling natif KO, ne pas continuer avec ce modèle."
print("tool-calling natif confirmé.")

Tool calls recus : [
  {
    "id": "call_ylmgb63s",
    "index": 0,
    "type": "function",
    "function": {
      "name": "get_weather",
      "arguments": "{\"city\":\"Paris\"}"
    }
  }
]
OK -- tool-calling natif confirmé.


## 6. Exposer le serveur via un tunnel public

In [35]:
USE_NGROK = True

if USE_NGROK:
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    NGROK_AUTHTOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")
    conf.get_default().auth_token = NGROK_AUTHTOKEN

    # host_header="localhost:11434" (rewrite) : Ollama rejette en 403 silencieux toute requete
    # dont le header Host ne correspond pas a localhost/127.0.0.1 (protection anti DNS-rebinding,
    # verification distincte du controle CORS Origin couvert par OLLAMA_ORIGINS).
    public_url = ngrok.connect(11434, "http", host_header="localhost:11434")
    print("URL publique ngrok (temporaire, valable tant que ce notebook tourne) :")
    print(public_url)
else:
    print("USE_NGROK=False -- cloudflared reste le tunnel actif.")
    # Tunnel via cloudflared (aucun compte requis)
    import subprocess, re, time
    
    !curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

    cloudflared_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:11434",
         "--http-host-header", "localhost:11434"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )

    public_url = None
    _t0 = time.time()
    for line in cloudflared_proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            break
        if time.time() - _t0 > 60:
            break

    if public_url:
        print("URL publique (temporaire, valable tant que ce notebook tourne) :")
        print(public_url)
    else:
        raise RuntimeError("URL cloudflared non trouvée dans les logs apres 60s -- verifier cloudflared_proc.stdout manuellement.")

URL publique ngrok (temporaire, valable tant que ce notebook tourne) :                              
NgrokTunnel: "https://iciness-clubhouse-spout.ngrok-free.dev" -> "http://localhost:11434"


## 7. où tourne le pipeline MIRO AI / l'accès Odoo

Dans le `.env` de `odoo-ia-prototype` (ou en variables d'environnement de la session), positionner temporairement :

```
LLM_PROVIDER=ollama
LOCAL_LLM_BASE_URL=<URL_TUNNEL_CI_DESSUS>/v1
LOCAL_LLM_MODEL=gpt-oss:20b
LOCAL_LLM_API_KEY=ollama
```


In [37]:
5+9

14